# Notebook 03: Adversarial Examples — PGD & C&W

## Why This Matters
FGSM is one gradient step. PGD takes many. That small change has outsized consequences: PGD-trained models are the current state of the art in adversarial robustness, PGD is the standard benchmark attack used to evaluate any defense, and the Madry et al. (2018) paper that introduced it for adversarial training is one of the most-cited papers in ML security.

C&W (Carlini & Wagner, 2017) takes a different angle: instead of maximizing misclassification for a fixed budget, it finds the *minimum distortion* needed to cause misclassification. C&W attacks are harder to pull off but find more efficient adversarial examples — and they're what's used to break defenses that claim to pass PGD evaluation.

Together, PGD and C&W define the threat model that serious robustness research is evaluated against.

## Structure
1. From FGSM to PGD: the projection step
2. PGD: algorithm, intuition, and the epsilon ball geometry
3. PGD as inner maximization: the Madry et al. minimax formulation
4. PGD-based adversarial training
5. C&W: minimum distortion as an optimization problem
6. Comparing FGSM, PGD, and C&W empirically
7. AutoAttack: reliable evaluation in practice

## What You'll Learn
- Why the projection step in PGD is necessary and what it enforces
- The minimax saddle point view of adversarial training
- Why C&W outperforms PGD on distortion efficiency
- How to use these attacks to honestly evaluate a defense
- Why PGD-AT (adversarial training) is still the dominant practical defense 6+ years after its publication

## Background

### The problem with FGSM

FGSM is a single gradient step. It moves $x$ in the direction that maximally increases the loss, scaled to budget $\epsilon$. This is efficient but weak — one step often doesn't reach the decision boundary for moderate $\epsilon$, especially in a high-dimensional space with a non-linear loss landscape.

More importantly, FGSM adversarial training has a failure mode that wasn't obvious until Madry et al. (2018) pointed it out: models trained against FGSM become resistant to FGSM specifically but remain vulnerable to stronger attacks. The model learns the texture of FGSM-specific perturbations rather than learning to be generally robust. This is a form of overfitting to a weak adversary.

### PGD: iterated FGSM with projection (Madry et al., 2018)

Projected Gradient Descent (PGD) extends FGSM by taking many small gradient steps, projecting back to the $\epsilon$-ball after each step:

$$x^{t+1} = \Pi_{\mathcal{B}(x, \epsilon)} \left( x^t + \alpha \cdot \text{sign}(\nabla_{x^t} \mathcal{L}(f(x^t), y)) \right)$$

where $\Pi_{\mathcal{B}(x, \epsilon)}$ is the projection operator that clips each coordinate to stay within $\epsilon$ of the original $x$, and $\alpha$ is a small step size (typically $\epsilon / \text{n\_steps}$ or $2.5\epsilon / \text{n\_steps}$).

The projection is what makes PGD principled: it guarantees the final adversarial example stays within the $\epsilon$-ball while the iterative gradient ascent finds a much better (more damaging) point within that ball than FGSM's single step.

PGD is initialized randomly within the $\epsilon$-ball (not at $x$ itself), which prevents the attack from getting stuck in a bad local optimum. Multiple random restarts are used for a thorough evaluation.

### The Madry et al. minimax formulation

The key contribution of Madry et al. wasn't just the attack — it was framing adversarial training as a minimax problem:

$$\min_\theta \mathbb{E}_{(x,y) \sim \mathcal{D}} \left[ \max_{\delta \in \mathcal{B}(0, \epsilon)} \mathcal{L}(f_\theta(x + \delta), y) \right]$$

The inner maximization — find the worst-case perturbation — is solved approximately by PGD. The outer minimization — find model parameters that minimize worst-case loss — is solved by SGD. Training alternates between these two.

This formulation is clean, principled, and gives you a rigorous notion of what "robustness" means: the model minimizes the worst-case loss within the $\epsilon$-ball. PGD-AT (PGD adversarial training) models trained this way remain the strongest practical defense six years later.

### C&W: minimum distortion as optimization (Carlini & Wagner, 2017)

While PGD asks "given a budget $\epsilon$, can I cause misclassification?", C&W asks "what is the minimum distortion needed to cause misclassification?". These are dual problems, but C&W's formulation finds more efficient adversarial examples.

C&W transforms the problem: instead of constraining the perturbation and maximizing misclassification, it adds the distortion as a regularization term in an unconstrained optimization:

$$\min_{\delta} \|\delta\|_2^2 + c \cdot f(x + \delta)$$

where $f$ is a loss function designed to be 0 when the attack fails and negative when it succeeds (Carlini & Wagner define a specific objective with this property). The constant $c$ trades off distortion against attack success — binary search over $c$ is used to find the minimum $c$ that achieves misclassification.

C&W also uses a change of variables ($\delta = \tanh(w) \cdot \epsilon - x$) to ensure the adversarial example is always in the valid pixel range without projection.

C&W was the attack that broke most of the 2016–2017 proposed defenses, including those that claimed to defeat PGD. Carlini & Wagner published it specifically to challenge the field's evaluation practices — their point was that defenses should be evaluated against the strongest known attack, and PGD at the time was not that attack.

### AutoAttack: reliable evaluation without parameter tuning

Both PGD and C&W require parameter choices ($\alpha$, number of steps, $c$, restarts) that affect how strong the attack is. An evaluator who isn't an adversarial ML expert might choose weak parameters and report misleading robustness numbers.

AutoAttack (Croce & Hein, 2020) addresses this: an ensemble of four complementary attacks (APGD-CE, APGD-DLR, FAB, Square) that doesn't require parameter tuning. It's parameter-free, deterministic, and has become the standard benchmark for evaluating robustness in the research community. The RobustBench leaderboard uses AutoAttack.

In [ ]:
%pip install -q torch torchvision matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"Device: {device}")

# Reuse MNIST setup from notebook 02
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
test_dataset = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_dataset = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True)
print(f"Data loaded: {len(train_dataset):,} train / {len(test_dataset):,} test")

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.drop  = nn.Dropout(0.25)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)


def evaluate(model, loader, device, n_batches=None):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if n_batches and i >= n_batches:
                break
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total


# Train a clean model to use as our attack target
model = MnistCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training clean model...")
for epoch in range(5):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        F.cross_entropy(model(x), y).backward()
        opt.step()
print(f"Clean model accuracy: {evaluate(model, test_loader, device):.4f}")

## 1. PGD: Algorithm and Geometry

The key addition over FGSM: the projection operator $\Pi_{\mathcal{B}(x, \epsilon)}$ that keeps every iterate within the $\epsilon$-ball.

Under $L_\infty$, this projection is simply elementwise clamping:
$$\Pi_{\mathcal{B}(x, \epsilon)}(x') = \text{clip}(x', x - \epsilon, x + \epsilon)$$

Under $L_2$, projection onto the ball is: normalize if the perturbation exceeds $\epsilon$:
$$\Pi_{\mathcal{B}(x, \epsilon)}^{L_2}(x') = x + \epsilon \cdot \frac{x' - x}{\max(\|x' - x\|_2, \epsilon)}$$

Random initialization within the $\epsilon$-ball prevents convergence to a bad local maximum.

In [ ]:
def pgd_attack(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    epsilon: float,
    alpha: float,
    n_steps: int,
    n_restarts: int = 1,
    norm: str = 'linf',
) -> torch.Tensor:
    """
    PGD attack (Madry et al., 2018).

    Args:
        epsilon:    L-inf or L-2 perturbation budget
        alpha:      step size (typically epsilon/n_steps * 2.5)
        n_steps:    number of gradient steps
        n_restarts: number of random restarts (use >1 for evaluation)
        norm:       'linf' or 'l2'

    Returns:
        best_x_adv: adversarial example with highest loss across all restarts
    """
    model.eval()
    best_x_adv = x.clone()
    best_loss  = torch.full((x.shape[0],), -float('inf'), device=x.device)

    for _ in range(n_restarts):
        # Random initialization within epsilon ball
        if norm == 'linf':
            delta = torch.zeros_like(x).uniform_(-epsilon, epsilon)
        else:  # l2
            delta = torch.randn_like(x)
            delta = delta / delta.view(delta.size(0), -1).norm(dim=1).view(-1, 1, 1, 1) * epsilon

        delta = torch.clamp(x + delta, x.min(), x.max()) - x  # stay in valid range
        delta.requires_grad_(True)

        for step in range(n_steps):
            loss = F.cross_entropy(model(x + delta), y, reduction='none')
            loss.sum().backward()

            with torch.no_grad():
                if norm == 'linf':
                    # Gradient sign step + L-inf projection
                    delta_new = delta + alpha * delta.grad.sign()
                    delta_new = torch.clamp(delta_new, -epsilon, epsilon)  # project to epsilon ball
                else:  # l2
                    # Normalized gradient step + L2 projection
                    grad_norm = delta.grad.view(delta.size(0), -1).norm(dim=1).view(-1, 1, 1, 1)
                    delta_new = delta + alpha * delta.grad / (grad_norm + 1e-8)
                    # Project onto L2 ball
                    delta_norm = delta_new.view(delta_new.size(0), -1).norm(dim=1).view(-1, 1, 1, 1)
                    delta_new = delta_new * torch.clamp(epsilon / delta_norm, max=1)

                delta_new = torch.clamp(x + delta_new, x.min(), x.max()) - x  # valid range
                delta = delta_new.requires_grad_(True)

        # Keep best adversarial examples (highest loss) across restarts
        with torch.no_grad():
            final_loss = F.cross_entropy(model(x + delta), y, reduction='none')
            improved = final_loss > best_loss
            best_x_adv = torch.where(improved.view(-1, 1, 1, 1), x + delta, best_x_adv)
            best_loss = torch.where(improved, final_loss, best_loss)

    return best_x_adv.detach()


def fgsm_attack(model, x, y, epsilon):
    x_adv = x.clone().detach().requires_grad_(True)
    F.cross_entropy(model(x_adv), y).backward()
    return torch.clamp(x_adv.detach() + epsilon * x_adv.grad.sign(), x.min(), x.max())


# Compare FGSM vs PGD on a single batch
x_b, y_b = next(iter(test_loader))
x_b, y_b = x_b.to(device), y_b.to(device)

epsilon = 0.3
x_fgsm = fgsm_attack(model, x_b, y_b, epsilon)
x_pgd  = pgd_attack(model, x_b, y_b, epsilon=epsilon, alpha=0.01, n_steps=40, n_restarts=1)

with torch.no_grad():
    acc_clean = (model(x_b).argmax(1) == y_b).float().mean().item()
    acc_fgsm  = (model(x_fgsm).argmax(1) == y_b).float().mean().item()
    acc_pgd   = (model(x_pgd).argmax(1) == y_b).float().mean().item()

print(f"ε = {epsilon}")
print(f"  Clean accuracy:  {acc_clean:.4f}")
print(f"  FGSM accuracy:   {acc_fgsm:.4f}  (attack success: {1-acc_fgsm:.4f})")
print(f"  PGD accuracy:    {acc_pgd:.4f}  (attack success: {1-acc_pgd:.4f})")
print()
print(f"PGD vs FGSM L-inf perturbation:")
print(f"  FGSM: {(x_fgsm - x_b).abs().max():.4f}  (should equal ε={epsilon})")
print(f"  PGD:  {(x_pgd - x_b).abs().max():.4f}  (bounded by ε={epsilon})")

In [ ]:
# Visualize PGD iterations: how loss evolves across steps
x_single = x_b[:1]
y_single = y_b[:1]

losses_per_step = []
preds_per_step  = []

delta = torch.zeros_like(x_single).uniform_(-epsilon, epsilon).requires_grad_(True)
alpha = 0.01
n_steps = 50

for step in range(n_steps):
    loss = F.cross_entropy(model(x_single + delta), y_single, reduction='none')
    losses_per_step.append(loss.item())

    with torch.no_grad():
        pred = model(x_single + delta).argmax(1).item()
    preds_per_step.append(pred)

    loss.sum().backward()
    with torch.no_grad():
        delta_new = delta + alpha * delta.grad.sign()
        delta_new = torch.clamp(delta_new, -epsilon, epsilon)
        delta_new = torch.clamp(x_single + delta_new, x_single.min(), x_single.max()) - x_single
        delta = delta_new.requires_grad_(True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
steps = range(n_steps)

ax1.plot(steps, losses_per_step, color='steelblue', linewidth=2)
ax1.set_ylabel('Cross-entropy loss')
ax1.set_title(f'PGD Attack Progress (ε={epsilon}, α={alpha})')
ax1.grid(True, alpha=0.3)

true_label = y_single.item()
colors = ['red' if p != true_label else 'green' for p in preds_per_step]
ax2.scatter(steps, preds_per_step, c=colors, s=15, zorder=3)
ax2.axhline(true_label, color='black', linestyle='--', alpha=0.5, label=f'True label: {true_label}')
ax2.set_ylabel('Model prediction')
ax2.set_xlabel('PGD step')
ax2.set_yticks(range(10))
ax2.legend()
ax2.grid(True, alpha=0.3)

first_wrong = next((i for i, p in enumerate(preds_per_step) if p != true_label), None)
if first_wrong:
    ax1.axvline(first_wrong, color='red', linestyle='--', alpha=0.5)
    ax2.axvline(first_wrong, color='red', linestyle='--', alpha=0.5, label=f'First mismatch: step {first_wrong}')
    print(f"First misclassification at step {first_wrong}")

plt.tight_layout()
plt.savefig('pgd_iterations.png', bbox_inches='tight')
plt.show()

## 2. PGD Adversarial Training (Madry et al.)

PGD-AT (Projected Gradient Descent Adversarial Training) is the minimax formulation in practice:

```
For each training batch (x, y):
  1. Inner maximization: run PGD to find x_adv = argmax_{δ ∈ B(x,ε)} L(f(x+δ), y)
  2. Outer minimization: update θ to minimize L(f(x_adv), y)
```

Key choices:
- **Training ε**: must match the ε at which you want robustness
- **Training steps**: typically 7–10 steps (more doesn't help much, less leads to FGSM-style overfitting)
- **Step size α**: `2.5 × ε / n_steps` is the standard recommendation
- **Random restarts**: 1 restart is standard during training (more is too expensive)

In [ ]:
def pgd_adversarial_training(
    model: nn.Module,
    train_loader,
    device,
    epsilon: float = 0.3,
    n_steps: int = 7,
    epochs: int = 5,
) -> nn.Module:
    """
    Madry et al. PGD adversarial training.
    Inner loop: 7-step PGD with α = 2.5ε/n_steps
    Outer loop: minimize loss on adversarial examples
    """
    alpha = 2.5 * epsilon / n_steps
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            # Inner maximization: find worst-case perturbation
            x_adv = pgd_attack(model, x, y, epsilon=epsilon, alpha=alpha, n_steps=n_steps)

            # Outer minimization: update on adversarial examples
            model.train()
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x_adv), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        clean_acc = evaluate(model, test_loader, device, n_batches=20)
        print(f"  Epoch {epoch+1}/{epochs} — clean: {clean_acc:.4f}")

    return model


torch.manual_seed(42)
model_pgd_at = MnistCNN().to(device)
print("Training with PGD adversarial training (Madry et al.)...")
print("(This takes ~3-5 minutes — inner PGD runs on every batch)")
model_pgd_at = pgd_adversarial_training(model_pgd_at, train_loader, device, epsilon=0.3, n_steps=7, epochs=5)
print(f"\nPGD-AT final clean accuracy: {evaluate(model_pgd_at, test_loader, device):.4f}")

In [ ]:
# Evaluate PGD-AT vs clean model at various attack strengths
epsilons_eval = [0.0, 0.1, 0.2, 0.3, 0.4]
n_eval_batches = 10

print("Evaluating clean model vs PGD-AT model under PGD attack (40 steps, 3 restarts)...")
print(f"{'ε':>6} {'Clean model':>14} {'PGD-AT model':>15} {'Robustness gain':>17}")
print("-" * 58)

clean_accs, pgdat_accs = [], []

for eps in epsilons_eval:
    if eps == 0:
        acc_clean = evaluate(model, test_loader, device, n_eval_batches)
        acc_pgdat  = evaluate(model_pgd_at, test_loader, device, n_eval_batches)
    else:
        alpha_eval = 2.5 * eps / 40
        correct_clean = correct_pgdat = total = 0
        for i, (x, y) in enumerate(test_loader):
            if i >= n_eval_batches:
                break
            x, y = x.to(device), y.to(device)
            x_adv_c = pgd_attack(model,        x, y, eps, alpha_eval, 40, n_restarts=3)
            x_adv_p = pgd_attack(model_pgd_at, x, y, eps, alpha_eval, 40, n_restarts=3)
            with torch.no_grad():
                correct_clean += (model(x_adv_c).argmax(1) == y).sum().item()
                correct_pgdat  += (model_pgd_at(x_adv_p).argmax(1) == y).sum().item()
                total += y.size(0)
        acc_clean = correct_clean / total
        acc_pgdat  = correct_pgdat / total

    clean_accs.append(acc_clean)
    pgdat_accs.append(acc_pgdat)
    print(f"{eps:>6.2f} {acc_clean:>14.4f} {acc_pgdat:>15.4f} {acc_pgdat - acc_clean:>+17.4f}")

## 3. C&W Attack: Minimum Distortion

C&W reformulates adversarial example generation as finding the *minimum perturbation* $\delta$ that causes misclassification. Instead of the loss landscape approach of PGD, it directly optimizes distortion subject to misclassification.

The C&W $L_2$ attack objective:

$$\min_{w} \|\tanh(w) - x\|_2^2 + c \cdot f(\tanh(w))$$

where the change of variables $\delta = \tanh(w) - x$ ensures valid pixel range without projection, and the adversarial objective $f$ is chosen to be:

$$f(x') = \max\left( \max_{j \neq t} Z(x')_j - Z(x')_t, \ -\kappa \right)$$

for targeted attack ($t$ = target class), where $Z(x')$ are the logits and $\kappa$ is a confidence margin. This is $\leq 0$ exactly when the attack succeeds with margin $\kappa$. Binary search over $c$ finds the smallest $c$ (minimum distortion weight) that achieves misclassification.

C&W typically finds adversarial examples with much smaller $L_2$ distortion than PGD for the same misclassification.

In [ ]:
def cw_l2_attack(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    c: float = 1.0,
    kappa: float = 0.0,
    n_steps: int = 100,
    lr: float = 0.01,
    targeted: bool = False,
    target_class: torch.Tensor = None,
) -> torch.Tensor:
    """
    Carlini & Wagner L2 attack (simplified version).

    Uses tanh change of variables to avoid box constraints.
    Optimizes distortion + c * adversarial_objective jointly.

    For a full implementation with binary search over c, see the original paper.
    """
    model.eval()

    # x is normalized; recover approximate [0,1] range for tanh trick
    x_min, x_max = x.min().item(), x.max().item()

    # Change of variables: x_adv = 0.5 * (tanh(w) + 1) scaled to [x_min, x_max]
    # Initialize w so that tanh(w) ≈ x
    x_scaled = (x - x_min) / (x_max - x_min + 1e-8) * 2 - 1  # [-1, 1]
    x_scaled = torch.clamp(x_scaled, -1 + 1e-6, 1 - 1e-6)     # avoid atanh inf
    w = torch.atanh(x_scaled).clone().detach().requires_grad_(True)

    optimizer = torch.optim.Adam([w], lr=lr)
    y_onehot = F.one_hot(y, num_classes=10).float()

    best_adv  = x.clone()
    best_dist = torch.full((x.size(0),), float('inf'), device=x.device)

    for _ in range(n_steps):
        # Reconstruct adversarial input from w
        x_adv = 0.5 * (torch.tanh(w) + 1) * (x_max - x_min) + x_min

        # L2 distortion
        distortion = (x_adv - x).view(x.size(0), -1).pow(2).sum(dim=1)

        # Adversarial objective f: negative when attack succeeds
        logits = model(x_adv)
        if not targeted:
            # Untargeted: want any class ≠ true class to be highest
            correct_logit = (logits * y_onehot).sum(dim=1)
            other_logit   = (logits * (1 - y_onehot) - y_onehot * 1e9).max(dim=1).values
            f_obj = torch.clamp(correct_logit - other_logit + kappa, min=0)
        else:
            # Targeted: want target_class to be highest
            target_onehot = F.one_hot(target_class, 10).float()
            target_logit  = (logits * target_onehot).sum(dim=1)
            other_logit   = (logits * (1 - target_onehot) - target_onehot * 1e9).max(dim=1).values
            f_obj = torch.clamp(other_logit - target_logit + kappa, min=0)

        loss = distortion.sum() + c * f_obj.sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track best (lowest distortion successful) adversarial examples
        with torch.no_grad():
            preds = logits.argmax(dim=1)
            attack_success = (preds != y) if not targeted else (preds == target_class)
            improved = attack_success & (distortion < best_dist)
            best_adv  = torch.where(improved.view(-1, 1, 1, 1), x_adv.detach(), best_adv)
            best_dist = torch.where(improved, distortion.detach(), best_dist)

    return best_adv.detach()


# Compare FGSM, PGD, and C&W on a single batch
x_b, y_b = next(iter(test_loader))
x_b, y_b = x_b[:32].to(device), y_b[:32].to(device)  # smaller for C&W speed

epsilon = 0.3
x_fgsm = fgsm_attack(model, x_b, y_b, epsilon)
x_pgd  = pgd_attack(model, x_b, y_b, epsilon=epsilon, alpha=0.01, n_steps=40)
x_cw   = cw_l2_attack(model, x_b, y_b, c=1.0, n_steps=100, lr=0.01)

with torch.no_grad():
    acc_clean = (model(x_b).argmax(1) == y_b).float().mean().item()
    acc_fgsm  = (model(x_fgsm).argmax(1) == y_b).float().mean().item()
    acc_pgd   = (model(x_pgd).argmax(1) == y_b).float().mean().item()
    acc_cw    = (model(x_cw).argmax(1) == y_b).float().mean().item()

def l_inf(x_adv, x):
    return (x_adv - x).abs().max().item()

def l2(x_adv, x):
    return (x_adv - x).view(x.size(0), -1).norm(dim=1).mean().item()

print("Attack comparison (batch of 32 examples):")
print(f"{'Attack':>8} {'Accuracy':>10} {'L∞ dist':>10} {'L2 dist (mean)':>16}")
print("-" * 50)
print(f"{'Clean':>8} {acc_clean:>10.4f} {'—':>10} {'—':>16}")
print(f"{'FGSM':>8} {acc_fgsm:>10.4f} {l_inf(x_fgsm, x_b):>10.4f} {l2(x_fgsm, x_b):>16.4f}")
print(f"{'PGD-40':>8} {acc_pgd:>10.4f} {l_inf(x_pgd, x_b):>10.4f} {l2(x_pgd, x_b):>16.4f}")
print(f"{'C&W':>8} {acc_cw:>10.4f} {l_inf(x_cw, x_b):>10.4f} {l2(x_cw, x_b):>16.4f}")
print()
print("C&W: similar attack success but with much smaller L2 distortion")
print("L∞ not constrained for C&W — it optimizes L2 directly")

In [ ]:
# Visual comparison: FGSM vs PGD vs C&W perturbations
idx = 0  # pick one example

def unnorm(t):
    return (t * 0.3081 + 0.1307).cpu().squeeze().numpy()

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("FGSM vs PGD vs C&W: Images and Perturbations", fontsize=12)

examples = [
    ('Clean', x_b, 'black'),
    ('FGSM', x_fgsm, 'steelblue'),
    ('PGD-40', x_pgd, 'darkorange'),
    ('C&W', x_cw, 'darkgreen'),
]

with torch.no_grad():
    preds = {
        'Clean': model(x_b[idx:idx+1]).argmax(1).item(),
        'FGSM': model(x_fgsm[idx:idx+1]).argmax(1).item(),
        'PGD-40': model(x_pgd[idx:idx+1]).argmax(1).item(),
        'C&W': model(x_cw[idx:idx+1]).argmax(1).item(),
    }

true_label = y_b[idx].item()
for col, (name, x_show, color) in enumerate(examples):
    img = unnorm(x_show[idx])
    axes[0, col].imshow(np.clip(img, 0, 1), cmap='gray')
    pred = preds[name]
    title_color = 'red' if pred != true_label else 'black'
    axes[0, col].set_title(f'{name}\npred: {pred}', color=title_color, fontweight='bold')
    axes[0, col].axis('off')

    if name != 'Clean':
        pert = (x_show[idx] - x_b[idx]).cpu().squeeze().numpy()
        vmax = max(abs(pert).max(), 0.01)
        im = axes[1, col].imshow(pert, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        linf = abs(pert).max()
        l2_val = np.sqrt((pert**2).sum())
        axes[1, col].set_title(f'Perturbation\nL∞={linf:.3f} L2={l2_val:.3f}', fontsize=8)
        plt.colorbar(im, ax=axes[1, col], fraction=0.046)
    else:
        axes[1, 0].axis('off')
        axes[1, 0].text(0.5, 0.5, f'True label: {true_label}', ha='center', va='center',
                        fontsize=12, transform=axes[1, 0].transAxes)
    axes[1, col].axis('off') if name == 'Clean' else None

plt.tight_layout()
plt.savefig('attack_comparison.png', bbox_inches='tight')
plt.show()

## 4. Evaluating Defenses Honestly

The core lesson from Athalye, Carlini & Wagner (2018): most proposed defenses were evaluated only against weak attacks. When adaptive attacks were designed that specifically accounted for the defense, the defenses collapsed.

**Principles for honest robustness evaluation:**

1. **Use strong attacks with multiple restarts.** FGSM is not a sufficient benchmark. PGD with 20+ steps and 5+ restarts is the minimum bar.

2. **Use adaptive attacks.** If your defense does anything differentiable (preprocessing, detection), the attacker can compute gradients through it. Evaluate with an attack that does.

3. **Report both clean and robust accuracy.** A defense that achieves 0% clean accuracy is trivially robust. The clean/robust tradeoff must be reported.

4. **Use AutoAttack for publication-level evaluation.** It's parameter-free, doesn't require attack expertise, and is the standard for RobustBench.

5. **Check for gradient masking.** Signs of gradient masking: loss decreases then increases as ε grows, PGD with more steps does *worse* than with fewer, random noise is nearly as effective as PGD. These are red flags that the gradient is being obscured, not that the defense works.

In [ ]:
# Diagnostic: check for gradient masking
# A robust model should show monotonically increasing attack success with more PGD steps
# A gradient-masked model may not

x_eval, y_eval = next(iter(test_loader))
x_eval, y_eval = x_eval[:64].to(device), y_eval[:64].to(device)
eps = 0.2

print("Gradient masking diagnostic:")
print("Robust model: attack success should increase (or plateau) with more steps")
print("Gradient masking: attack success would decrease with more steps (suspicious!)")
print()
print(f"{'Steps':>8} {'Clean model (attack success)':>30} {'PGD-AT model (attack success)':>32}")
print("-" * 75)

for n_steps in [1, 5, 10, 20, 40]:
    alpha = 2.5 * eps / n_steps
    x_adv_c = pgd_attack(model,        x_eval, y_eval, eps, alpha, n_steps)
    x_adv_p = pgd_attack(model_pgd_at, x_eval, y_eval, eps, alpha, n_steps)
    with torch.no_grad():
        succ_c = (model(x_adv_c).argmax(1) != y_eval).float().mean().item()
        succ_p = (model_pgd_at(x_adv_p).argmax(1) != y_eval).float().mean().item()
    print(f"{n_steps:>8} {succ_c:>30.4f} {succ_p:>32.4f}")

print()
print("Ideal: attack success increases with steps until saturation")
print("Red flag: attack success DECREASES with more steps → gradient masking suspected")

In [ ]:
# Final robustness comparison: clean, FGSM-AT, PGD-AT evaluated under PGD-40
fig, ax = plt.subplots(figsize=(9, 5))

epsilons_plot = [0.0, 0.1, 0.2, 0.3]
clean_rob, pgdat_rob = [], []

for eps in epsilons_plot:
    if eps == 0:
        clean_rob.append(evaluate(model, test_loader, device, 10))
        pgdat_rob.append(evaluate(model_pgd_at, test_loader, device, 10))
    else:
        alpha = 2.5 * eps / 40
        cc = cp = total = 0
        for i, (x, y) in enumerate(test_loader):
            if i >= 10:
                break
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                cc += (model(pgd_attack(model, x, y, eps, alpha, 40)).argmax(1) == y).sum().item()
                cp += (model_pgd_at(pgd_attack(model_pgd_at, x, y, eps, alpha, 40)).argmax(1) == y).sum().item()
                total += y.size(0)
        clean_rob.append(cc / total)
        pgdat_rob.append(cp / total)

ax.plot(epsilons_plot, clean_rob, 'o-', color='steelblue', linewidth=2, label='Standard training', markersize=7)
ax.plot(epsilons_plot, pgdat_rob, 's-', color='darkorange', linewidth=2, label='PGD Adversarial Training', markersize=7)
ax.axhline(0.1, color='gray', linestyle=':', alpha=0.6, label='Random chance')
ax.fill_between(epsilons_plot, clean_rob, pgdat_rob, alpha=0.1, color='green')

ax.set_xlabel('Epsilon (L∞ budget)')
ax.set_ylabel('Accuracy under PGD-40 attack')
ax.set_title('Standard vs PGD Adversarial Training — Robustness under PGD-40')
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pgd_at_comparison.png', bbox_inches='tight')
plt.show()

## Summary

```
Attack       Steps   Constraint   Strength   Distortion
─────────────────────────────────────────────────────────
FGSM         1       L∞           Weak       Maximum at ε
PGD          k       L∞ or L2     Strong     Bounded by ε
C&W          many    L2           Strongest  Minimized
```

**Key facts:**
- PGD = iterated FGSM + projection back to ε-ball after each step
- Random initialization within ε-ball prevents local optima; multiple restarts for thoroughness
- Madry et al. minimax: inner max (PGD) finds worst-case δ; outer min (SGD) updates θ against it
- PGD-AT is still the dominant practical defense — no published defense has definitively surpassed it
- C&W minimizes distortion directly; finds more efficient adversarial examples than PGD
- Monotonically increasing attack success with more PGD steps is the sanity check for any defense claim
- Always evaluate with adaptive attacks — computing gradients through the defense

**Next:** Notebook 04 covers adversarial defenses in depth — adversarial training variants, certified defenses (randomized smoothing), and a systematic look at what has and hasn't worked and why.